In [3]:
import pandas as pd


transactions = pd.read_parquet(
    "../data/bronze/transactions.parquet"
)

transactions.head()


,gitOutlet_ID,Year,Month,Distributor_ID,SKU_ID,Volume_Liters,Total_Bill_Value
0,OUT_19886,2024,12,DIST_S_02,SKU_10,5.897879,2177.632359
1,OUT_00837,2024,2,DIST_W_01,SKU_03,20.697364,7244.084814
2,OUT_15438,2025,12,DIST_NW_01,SKU_02,55.101801,13959.108790
3,OUT_12992,2025,1,DIST_C_01,SKU_07,24.063953,15641.548770
4,OUT_12334,2025,5,DIST_C_02,SKU_04,47.769665,15525.158660


In [5]:
transactions.columns

Index(['gitOutlet_ID', 'Year', 'Month', 'Distributor_ID', 'SKU_ID',
       'Volume_Liters', 'Total_Bill_Value'],
      dtype='str')

In [24]:
#Remove Duplicate Rows
transactions = transactions.drop_duplicates(
    subset=[
        "gitOutlet_ID",
        "Year",
        "Month",
        "Distributor_ID",
        "SKU_ID"
    ]
)

In [25]:
#Remove Missing Values
transactions = transactions.dropna(
    subset=[
        "gitOutlet_ID",
        "Distributor_ID",
        "SKU_ID",
        "Volume_Liters"
    ]
)

In [26]:
#Remove Invalid Sales Volumes
transactions = transactions[
    transactions["Volume_Liters"] >= 0
]

In [27]:
#Remove Invalid Bill Values
transactions = transactions[
    transactions["Total_Bill_Value"] >= 0
]

In [28]:
#Remove Invalid Year
transactions = transactions[
    (transactions["Year"] >= 2020) &
    (transactions["Year"] <= 2026)
]

In [29]:
#Remove Invalid Month
transactions = transactions[
    (transactions["Month"] >= 1) &
    (transactions["Month"] <= 12)
]

In [30]:
#Save Clean Transactions
transactions.to_parquet(
    "../data/silver/clean_transactions.parquet",
    index=False
)

In [13]:
transactions.shape

(1039959, 7)

In [14]:
holidays = pd.read_parquet(
    "../data/bronze/holidays.parquet"
)

holidays.head()

,Date,Holiday_Name,Holiday_Type
0,2023-01-06T00:00:00Z,Duruthu Full Moon Poya Day,Public
1,2023-01-15T00:00:00Z,Tamil Thai Pongal Day,Public
2,2023-01-16T00:00:00Z,Additional Holiday in lieu of Tamil Thai Ponga...,Public
3,2023-02-03T00:00:00Z,Additional Half Holiday in lieu of the Indepen...,Public
4,2023-02-04T00:00:00Z,National Day,Public


In [15]:
holidays.columns

Index(['Date', 'Holiday_Name', 'Holiday_Type'], dtype='str')

In [31]:
#Convert Date Column
holidays["Date"] = pd.to_datetime(
    holidays["Date"],
    errors="coerce"
)

In [32]:
#Remove Invalid Dates
holidays = holidays[
    holidays["Date"].notnull()
]

In [33]:
#Remove Duplicate Rows
holidays = holidays.drop_duplicates()

In [34]:
#Remove Missing Holiday Names
holidays = holidays.dropna(
    subset=["Holiday_Name"]
)

In [35]:
#Remove Missing Holiday Types
holidays = holidays.dropna(
    subset=["Holiday_Type"]
)

In [36]:
#Save Clean Holiday Dataset
holidays.to_parquet(
    "../data/silver/clean_holidays.parquet",
    index=False
)

In [37]:
coordinates = pd.read_parquet(
    "../data/bronze/outletCoordinates.parquet"
)

coordinates.head()

,Outlet_ID,Latitude,Longitude
0,OUT_00001,7.089846,79.979055
1,OUT_00002,7.000558,80.012422
2,OUT_00003,6.806170,79.854547
3,OUT_00004,6.703533,79.806919
4,OUT_00005,7.186878,79.869831


In [38]:
coordinates.columns

Index(['Outlet_ID', 'Latitude', 'Longitude'], dtype='str')

In [39]:
#Remove Duplicate Outlets
coordinates = coordinates.drop_duplicates(
    subset=["Outlet_ID"]
)

In [40]:
#Remove Missing Values
coordinates = coordinates.dropna(
    subset=[
        "Outlet_ID",
        "Latitude",
        "Longitude"
    ]
)


In [41]:
#Convert Coordinates to Numeric
coordinates["Latitude"] = pd.to_numeric(
    coordinates["Latitude"],
    errors="coerce"
)

coordinates["Longitude"] = pd.to_numeric(
    coordinates["Longitude"],
    errors="coerce"
)

In [42]:
#Remove Invalid Latitude
coordinates = coordinates[
    (coordinates["Latitude"] >= 5) &
    (coordinates["Latitude"] <= 10)
]

In [43]:
#Remove Invalid Longitude
coordinates = coordinates[
    (coordinates["Longitude"] >= 79) &
    (coordinates["Longitude"] <= 82)
]

In [44]:
#Save Clean Dataset
coordinates.to_parquet(
    "../data/silver/clean_outletCoordinates.parquet",
    index=False
)

In [45]:
coordinates.shape

(19760, 3)

In [46]:
import pandas as pd

outlets = pd.read_parquet(
    "../data/bronze/outletMaster.parquet"
)

outlets.head()

,Outlet_ID,Outlet_Size,Cooler_Count,Outlet_Type
0,OUT_00001,Medium,1,Grocry
1,OUT_00002,Small,0,Hotel
2,OUT_00003,Small,1,Pharmacy
3,OUT_00004,Medium,2,Pharmacy
4,OUT_00005,Medium,2,Kiosk


In [47]:
outlets.columns

Index(['Outlet_ID', 'Outlet_Size', 'Cooler_Count', 'Outlet_Type'], dtype='str')

In [48]:
#Remove Duplicate Outlets
outlets = outlets.drop_duplicates(
    subset=["Outlet_ID"]
)

In [49]:
#Remove Missing Outlet_ID
outlets = outlets.dropna(
    subset=["Outlet_ID"]
)

In [50]:
#Remove Negative Cooler Counts
outlets = outlets[
    outlets["Cooler_Count"] >= 0
]

In [51]:
#Fill Missing Outlet_Size
outlets["Outlet_Size"] = outlets[
    "Outlet_Size"
].fillna("Unknown")

In [52]:
#Fill Missing Outlet_Type
outlets["Outlet_Type"] = outlets[
    "Outlet_Type"
].fillna("Unknown")

In [53]:
#Remove Extra Spaces
outlets["Outlet_Size"] = outlets[
    "Outlet_Size"
].str.strip()

outlets["Outlet_Type"] = outlets[
    "Outlet_Type"
].str.strip()

In [54]:
#Save Clean Dataset
outlets.to_parquet(
    "../data/silver/clean_outlet_master.parquet",
    index=False
)

In [55]:
outlets.shape

(20000, 4)

In [56]:
import pandas as pd

seasonality = pd.read_parquet(
    "../data/bronze/seasonality.parquet"
)

seasonality.head()

,Distributor_ID,Year,Month,Seasonality_Index
0,DIST_W_01,2023,1,Moderate
1,DIST_W_01,2023,2,Moderate
2,DIST_W_01,2023,3,Moderate
3,DIST_W_01,2023,4,Favorable
4,DIST_W_01,2023,5,Un-Favorable


In [58]:
seasonality.columns

Index(['Distributor_ID', 'Year', 'Month', 'Seasonality_Index'], dtype='str')

In [59]:
#Remove Duplicate Rows
seasonality = seasonality.drop_duplicates(
    subset=[
        "Distributor_ID",
        "Year",
        "Month"
    ]
)

In [61]:
#Remove Missing Values
seasonality = seasonality.dropna(
    subset=[
        "Distributor_ID",
        "Year",
        "Month",
        "Seasonality_Index"
    ]
)

In [63]:
#Remove Invalid Years
seasonality = seasonality[
    (seasonality["Year"] >= 2020) &
    (seasonality["Year"] <= 2026)
]

In [64]:
#Remove Invalid Months
seasonality = seasonality[
    (seasonality["Month"] >= 1) &
    (seasonality["Month"] <= 12)
]

In [66]:
seasonality["Seasonality_Index"] = pd.to_numeric(
    seasonality["Seasonality_Index"],
    errors="coerce"
)

In [67]:
#Remove Invalid Seasonality Values
seasonality = seasonality[
    seasonality["Seasonality_Index"] >= 0
]

In [68]:
#Save Clean Dataset
seasonality.to_parquet(
    "../data/silver/clean_seasonality.parquet",
    index=False
)

In [69]:
seasonality.shape

(0, 4)